# PDPA Assistant — Week 1: Build the core RAG pipeline

By the end of this notebook you will have a working assistant that answers questions about
Singapore's **Personal Data Protection Act (PDPA)**, using only official documents, with citations.

**The 6 steps we build:**

| Step | What happens | RAG part |
|---|---|---|
| 1 | Load the PDF documents into text | Preparation |
| 2 | Split the text into small chunks | Preparation |
| 3 | Turn each chunk into numbers (embeddings) and save them to disk with NumPy | Preparation |
| 4 | Find the chunks closest to a question | **R**etrieval |
| 5 | Put those chunks into a prompt | **A**ugmentation |
| 6 | The AI writes an answer with citations | **G**eneration |

**How to use this notebook:** run one cell at a time (Shift + Enter), read the explanation above it,
and look at the output before moving on. Understanding each output is the real learning.

## Before you start (checklist)

1. **Ollama is installed and running** (download from ollama.com). In a terminal, run:
   ```
   ollama pull nomic-embed-text
   ollama pull llama3.2
   ```
2. **Your folder looks like this:**
   ```
   pdpa-assistant/
   ├── pdpa_week1.ipynb      ← this notebook
   ├── sources.csv           ← list of your documents
   └── documents/            ← put your PDPA PDFs here
       ├── pdpa_act.pdf
       ├── key_concepts.pdf
       └── ...
   ```
3. **The file names in `sources.csv` match the PDF names** in the `documents` folder.

## Step 0 — Install the libraries (run once)

- `numpy` — fast maths on lists of numbers; stores and searches our embeddings (usually already in Anaconda)
- `pypdf` — reads text out of PDF files
- `ollama` — lets Python talk to the AI models running on your computer

In [ ]:
%pip install numpy pypdf ollama

## Settings

All the values you might want to change later are in one place.
When you experiment in Week 2 (e.g. different chunk sizes), this is the only cell you edit.

In [ ]:
from pathlib import Path

# The folder this notebook is running in (your project folder)
PROJECT_FOLDER = Path.cwd()

DOCS_FOLDER = PROJECT_FOLDER / "documents"     # where your PDFs are
SOURCES_FILE = PROJECT_FOLDER / "sources.csv"  # titles, links and dates of your documents
DB_FOLDER = PROJECT_FOLDER / "pdpa_db"         # where the embeddings are saved

EMBED_MODEL = "nomic-embed-text"     # turns text into numbers
CHAT_MODEL = "llama3.2"              # writes the answers

CHUNK_SIZE = 300                     # words per chunk
CHUNK_OVERLAP = 50                   # words shared between neighbouring chunks
TOP_K = 4                            # how many chunks to retrieve per question

# Quick checks
print("Notebook folder:", PROJECT_FOLDER)
print("Documents folder found:", DOCS_FOLDER.exists())
print("sources.csv found:", SOURCES_FILE.exists())

Let's check the PDFs are where we expect them.

In [ ]:
pdf_files = sorted(DOCS_FOLDER.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF file(s):")
for path in pdf_files:
    print("  -", path.name)

if not pdf_files:
    print("No PDFs found. Put your PDPA documents in the 'documents' folder and run this cell again.")

Now read `sources.csv`. It stores each document's **title, link and download date**,
so our answers can show proper citations instead of just file names.

In [ ]:
import csv

sources = {}
with open(SOURCES_FILE, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        sources[row["file_name"]] = row

for path in pdf_files:
    if path.name in sources:
        print(f"OK       {path.name}  →  {sources[path.name]['title']}")
    else:
        print(f"MISSING  {path.name} is not listed in sources.csv (it will still work, but citations show the file name)")

---
## Step 1 — Load the documents

`pypdf` reads each page of each PDF and gives us its text.

PDF text is often messy: extra spaces, line breaks in the middle of sentences, words split with a hyphen
across lines. The `clean_text` function tidies this up. We also **keep the page number** for each page —
we'll need it for citations later.

In [ ]:
import re
from pypdf import PdfReader


def clean_text(text):
    """Tidy up text extracted from a PDF."""
    text = text.replace("\u00a0", " ")          # odd 'non-breaking' spaces → normal spaces
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)   # re-join words split across lines: 'organi-\nsation'
    text = re.sub(r"\s+", " ", text)              # many spaces/newlines → one space
    return text.strip()


def load_pdf(path):
    """Return a list of pages: each page is a dictionary with source, page number and text."""
    reader = PdfReader(path)
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = clean_text(page.extract_text() or "")
        if text:                                   # skip empty pages
            pages.append({"source": path.name, "page": page_number, "text": text})
    return pages


all_pages = []
for path in pdf_files:
    pages = load_pdf(path)
    all_pages.extend(pages)
    print(f"{path.name}: {len(pages)} pages with text")

print(f"\nTotal: {len(all_pages)} pages")

**Always look at your data.** Print part of one page to check the text came out readable.
If you see garbled characters or no text at all, that PDF may be a scanned image (which needs OCR — skip it for now).

In [ ]:
example = all_pages[0]
print(f"Source: {example['source']}, page {example['page']}\n")
print(example["text"][:800])

---
## Step 2 — Split into chunks

A page can be long and cover several topics. We split each page into **chunks of about 300 words**.

**Why the overlap?** If an important sentence sits right at the border between two chunks, it could be cut in half.
With a 50-word overlap, the end of one chunk is repeated at the start of the next, so nothing important is lost.

Each chunk remembers its **source file and page number** (this extra information is called *metadata*).

In [ ]:
def chunk_page(page, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split one page into overlapping chunks of words."""
    words = page["text"].split()
    chunks = []
    step = size - overlap
    for start in range(0, len(words), step):
        piece = " ".join(words[start:start + size])
        chunks.append({"text": piece, "source": page["source"], "page": page["page"]})
        if start + size >= len(words):            # reached the end of the page
            break
    return chunks


all_chunks = []
for page in all_pages:
    all_chunks.extend(chunk_page(page))

print(f"{len(all_pages)} pages → {len(all_chunks)} chunks")

Look at one chunk and its metadata:

In [ ]:
example = all_chunks[0]
print(f"Source: {example['source']} | Page: {example['page']} | Words: {len(example['text'].split())}\n")
print(example["text"][:600], "...")

---
## Step 3 — Embed the chunks and save them

**Embedding** = turning text into a list of numbers that represents its *meaning*.
Texts with similar meanings get similar numbers — so "data leak" and "data breach" end up close together.

We first test the embedding model on one sentence, so you can see what an embedding looks like.

In [ ]:
import ollama


def embed_texts(texts):
    """Turn a list of texts into a list of embeddings (one list of numbers per text)."""
    response = ollama.embed(model=EMBED_MODEL, input=texts)
    return response["embeddings"]


test = embed_texts(["An organisation must protect personal data in its possession."])[0]
print(f"One sentence became {len(test)} numbers. The first five are:")
print([round(x, 4) for x in test[:5]])

Now we embed **every** chunk and keep the results as a NumPy **matrix**: one row of 768 numbers per chunk.
This matrix is our "vector index" — the job a vector database like ChromaDB would do.

- We embed in **batches** of 32 chunks at a time — faster than one by one.
- Each row is **normalised** (scaled to length 1). That makes searching later a single, fast multiplication.
- Everything is saved in the `pdpa_db` folder: `embeddings.npy` (the numbers) and `chunks.json`
  (the text, source and page of each chunk), so the scoring notebook can reuse it without re-embedding.
- Running this cell again rebuilds everything from scratch (useful after adding documents or changing chunk size).

This can take a few minutes the first time.

In [ ]:
import json
import numpy as np

BATCH_SIZE = 32
vectors = []
for start in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[start:start + BATCH_SIZE]
    vectors.extend(embed_texts([c["text"] for c in batch]))
    print(f"Embedded {min(start + BATCH_SIZE, len(all_chunks))} / {len(all_chunks)} chunks")

EMBEDDINGS = np.array(vectors, dtype=np.float32)
EMBEDDINGS = EMBEDDINGS / np.linalg.norm(EMBEDDINGS, axis=1, keepdims=True)   # normalise each row

DB_FOLDER.mkdir(exist_ok=True)
np.save(DB_FOLDER / "embeddings.npy", EMBEDDINGS)
with open(DB_FOLDER / "chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False)

print(f"\nDone. Saved {EMBEDDINGS.shape[0]} chunks × {EMBEDDINGS.shape[1]} numbers to {DB_FOLDER}")

---
## Step 4 — Retrieve: find the chunks closest to a question

1. Embed the question with the **same** model, and normalise it.
2. Multiply the chunk matrix by the question vector. Because every row was normalised, this gives the
   **cosine similarity** between the question and every chunk at once (1.0 = identical meaning, near 0 = unrelated).
3. Sort the scores and keep the `TOP_K` best chunks.

In [ ]:
def retrieve(question, top_k=TOP_K):
    """Return the top_k chunks most similar in meaning to the question."""
    q = np.array(embed_texts([question])[0], dtype=np.float32)
    q = q / np.linalg.norm(q)

    scores = EMBEDDINGS @ q                      # similarity to every chunk at once
    best = np.argsort(scores)[::-1][:top_k]      # positions of the highest scores

    found = []
    for i in best:
        chunk = all_chunks[i]
        found.append({
            "text": chunk["text"],
            "source": chunk["source"],
            "page": chunk["page"],
            "similarity": round(float(scores[i]), 3),
        })
    return found


def source_title(file_name):
    """Readable title from sources.csv, or the file name if it isn't listed."""
    info = sources.get(file_name)
    return info["title"] if info and info.get("title") else file_name

**Try it.** Notice the question says *"data leak"*, while the documents say *"data breach"* —
retrieval should still find the right pages, because it searches by meaning.

Read the retrieved chunks carefully: **are they actually relevant?** Most problems in RAG are retrieval problems.

In [ ]:
question = "What should a company do if there is a data leak?"

for c in retrieve(question):
    print(f"[similarity {c['similarity']}]  {source_title(c['source'])}, page {c['page']}")
    print("   ", c["text"][:200], "...\n")

---
## Step 5 — Augment: build the prompt

We give the AI:
- **rules** (use only these sources, cite them, say "I don't know" if the answer isn't there),
- the **retrieved chunks**, each numbered so the AI can cite them as [1], [2]...,
- the **question**.

Let's print a prompt so you can see exactly what the AI receives — there's no magic, it's just text.

In [ ]:
def build_prompt(question, found):
    context_parts = []
    for number, c in enumerate(found, start=1):
        context_parts.append(f"[{number}] ({source_title(c['source'])}, page {c['page']})\n{c['text']}")
    context = "\n\n".join(context_parts)

    return f"""You are an assistant that explains Singapore's Personal Data Protection Act (PDPA).

Rules:
1. Answer using ONLY the numbered sources below.
2. After each fact, cite the source number in square brackets, for example [1].
3. If the sources do not contain the answer, reply exactly: "I don't have information on that in my PDPA documents."
4. Use simple, clear language that a small business owner can understand.

Sources:
{context}

Question: {question}
Answer:"""


print(build_prompt(question, retrieve(question))[:1500], "...")

---
## Step 6 — Generate: the AI writes the answer

`temperature = 0` makes the AI's answers more consistent (less "creative") — what we want for compliance questions.

Notice the **disclaimer is added by our code**, not by the AI. That guarantees it appears on every answer —
we don't rely on the AI remembering to include it.

In [ ]:
DISCLAIMER = "This is general information, not legal advice."


def generate(prompt):
    response = ollama.chat(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )
    return response["message"]["content"]


def ask(question):
    """The full RAG pipeline: retrieve → build prompt → generate → show answer with sources."""
    found = retrieve(question)
    answer = generate(build_prompt(question, found))

    print("QUESTION:", question)
    print("\nANSWER:\n" + answer)
    print("\nSOURCES:")
    for number, c in enumerate(found, start=1):
        print(f"  [{number}] {source_title(c['source'])}, page {c['page']} (similarity {c['similarity']})")
    print("\n" + DISCLAIMER)

## Try it!

Run each question and check: **Is the answer correct? Do the citations point to the right pages?**
Open the PDF and check one or two citations yourself.

In [ ]:
ask("When must an organisation notify the PDPC about a data breach?")

In [ ]:
ask("Can a shop collect a customer's full NRIC number for a lucky draw?")

In [ ]:
ask("Do we need to appoint a Data Protection Officer?")

### Test the "I don't know" rule

These questions are **not** about the PDPA. A good assistant should refuse instead of making something up.

In [ ]:
ask("What is the GST rate in Singapore?")

In [ ]:
ask("Who won the football World Cup in 2022?")

### Your turn

Write your own questions below. Try:
- the same question in different words (does retrieval still work?)
- a question you know the answer to (is the answer right?)
- a tricky one where two documents say different things

In [ ]:
ask("Type your own question here")

---
## What you built this week

✅ Loaded official PDPA documents and cleaned the text
✅ Split them into overlapping chunks with source and page metadata
✅ Embedded them and saved a NumPy vector index to disk
✅ Retrieved the most relevant chunks by meaning
✅ Generated cited answers with an "I don't know" rule and a guaranteed disclaimer

**Write down (for interviews and your README):**
- Which questions worked well? Which didn't, and why — was it retrieval (wrong chunks) or generation (right chunks, bad answer)?
- Any messy PDF text you had to deal with?

## Next week (Week 2)

1. Build a **test set** of 20–30 questions with the correct document and page for each.
2. Write a small loop that measures **how often retrieval finds the right page** → your accuracy score.
3. Try **one improvement** (e.g. chunk size 150 vs 300, or `TOP_K` 3 vs 6) and compare scores.